# Kopru Adreslenebilirligi — Asama A + B"Dongu icinde latent adresli bellek" tasariminin **oncul** ve **tasiyici** varsayimlarini,her biri **farkli bir eyleme** cikacak sekilde sinar.## Iddia zinciri1. Cok-hop basarisizligi bir **adresleme** problemidir: hop-2'nin anahtari (ara varlik)   girdide token olarak yoktur, token-adresli bellek ona ulasamaz.2. Ara varlik yine de **latent durumda** temsil edilir.3. Ve **cevaptan yeterince once** temsil edilir — arada is yapilabilecek bir pencere vardir.## Karar agaci — bes sonuc, bes ayri eylem| Sonuc | Anlam | Eylem ||---|---|---|| **ONCUL YANLIS** | Kopruyu bedava vermek yardim etmiyor | **DUR.** Darbogaz adresleme degil || **IPUCU KULLANILAMIYOR** | Model verilen kopruyu hicbir formatta kullanamiyor | **Belirsiz.** Asama A gecersiz, once bunu coz || **ADRES YOK** | Ezberleyemeyen, tam denetimli prob bile kopruyu cikaramiyor | **Kapsam degisir.** Lookup degil, egitim problemi || **KISAYOL** | Kopru, cevaptan once cozulmuyor | Model `e1->e3` kisayolu kullaniyor; ara adim yok || **PENCERE DAR** | Kopru once cozuluyor ama aralik is yapilamayacak kadar dar | Adres var, kullanilabilir zaman yok || **GEC** | Kopru erken cozuluyor, pencere genis | **Deney 2:** donguye bagla, merdiven testi |---## Hakem raporu sonrasi duzeltmeler| # | Sorun | Duzeltme ||---|---|---|| M1 | Kapi `dogru - rastgele` kullaniyordu; rastgele ipucu **notr degil dusmanca**, tek basina esigi doldurabilirdi | Kapi artik **`dogru - ipucusuz`**. Rastgele ipucu **saglamlik kontrolu** (model ipucunu okuyor mu) || M2 | "Kapi" sadece yazdiriliyordu; sonraki 5 hucre yine calisiyordu (~40 dk bosa) | `require_A()` — Asama B hucreleri **durur** || M6 | Kucuk fark "ipucu kullanamiyor" da olabilirdi -> **sahte DUR** | **Iki ipucu formati** denenir + filtre garantili **%100 tavan** teshis olarak raporlanir || M3 | `L_kopru < L_cevap` karari, 1 katmanlik ise yaramaz pencerede de geciyordu | **Pencere genisligi** karara girdi (`MIN_WINDOW_FRAC`) || M4 | Kopru probu ezberlemeye karsi korunuyordu, cevap probu **korunmuyordu** | **Iki ayri bolme**: kopru varligina gore ve cevap varligina gore || M5 | Varlik hedef uzayi keyfi olarak son katmandi, test edilmemisti | Hedef katman **val'da secilen bir hiperparametre** (lambda gibi) || M7 | Tarih/sayi varliklari gomme uzayinda ayirt edilemez, egrileri bozuyordu | Tarih/sayi varlikli sorular **elenir** || K1 | Rastgele ipucu gercek kopruyle **cakisabiliyordu** | String farkliligi garanti || K2 | `raw["answer"]`, `per_item_none` olu; `chosen_lam` hic raporlanmiyordu | Olu kod silindi; secilen hiperparametreler raporlanir || K3 | Varlik kodlamada tum katmanlar ayriliyor, biri kullaniliyordu | Yalnizca aday hedef katmanlar || K4 | Kucuk `n_train`'de uyari yoktu | Esik altinda uyari |> **Runtime > Change runtime type > T4 GPU**

## 1. Kurulum ve ayarlar

In [ ]:
!pip -q install -U transformers datasets accelerateimport torchprint("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())if torch.cuda.is_available():    p = torch.cuda.get_device_properties(0)    print("gpu:", p.name, f"{p.total_memory/1e9:.1f} GB")else:    print("GPU YOK -> Runtime > Change runtime type > T4 GPU")

In [ ]:
MODEL_ID = "Qwen/Qwen3-1.7B"# Alternatifler: "Qwen/Qwen3-0.6B", "Qwen/Qwen2.5-1.5B", "meta-llama/Llama-3.2-1B"MAX_ITEMS = 4000      # bilgi filtresine sokulacak aday soruPCA_DIM   = 256LAMBDAS   = (1.0, 10.0, 100.0, 1000.0, 10000.0)TGT_FRACS = (0.35, 0.65, 1.0)   # varlik gomme icin aday katmanlar (derinlik orani)# --- Asama A kapisi (M1) ---GATE_GAIN     = 0.15   # (dogru ipucu) - (ipucusuz) bu kadar puan gecmeliHINT_CEILING  = 0.60   # dogru-ipucu dogrulugu bunun altindaysa: IPUCU KULLANILAMIYOR (M6)# --- Asama B kararlari ---BRIDGE_CI_MAX    = 0.45  # kopru GA ust siniri bunun altinda olmaliMIN_WINDOW_FRAC  = 0.15  # (L_cevap - L_kopru) / katman sayisi (M3)MIN_KEPT, MIN_TRAIN = 150, 120GEN_BATCH, ENC_BATCH, N_BOOT, SEED = 32, 64, 2000, 0USE_DRIVE, DRIVE_DIR = True, "/content/drive/MyDrive/kopru"

In [ ]:
import os, json, re, string, randomimport numpy as np, torchfrom collections import defaultdictrandom.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)if USE_DRIVE:    from google.colab import drive; drive.mount("/content/drive"); WORK = DRIVE_DIRelse:    WORK = "/content/kopru"os.makedirs(WORK, exist_ok=True)MSLUG = MODEL_ID.split("/")[-1].replace(".", "-")STAMP = f"{MSLUG}__i{MAX_ITEMS}"def cpath(n): return os.path.join(WORK, f"{STAMP}__{n}")def cached(n): return os.path.exists(cpath(n))print("dizin:", WORK, "| damga:", STAMP)

## 2. Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLMdevice = "cuda" if torch.cuda.is_available() else "cpu"tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)if tok.pad_token is None: tok.pad_token = tok.eos_tokentok.padding_side = "left"model = AutoModelForCausalLM.from_pretrained(    MODEL_ID, torch_dtype=torch.float16, trust_remote_code=True).to(device)model.eval()N_LAYERS, H = model.config.num_hidden_layers, model.config.hidden_sizeLAYERS    = list(range(N_LAYERS + 1))TGT_LAYERS = sorted({int(round(f * N_LAYERS)) for f in TGT_FRACS})print(f"{MODEL_ID}: {N_LAYERS} katman, hidden={H}")print("varlik gommesi icin aday hedef katmanlar (val'da secilecek):", TGT_LAYERS)with torch.no_grad():    _h = model(**tok(["The capital of France is Paris."], return_tensors="pt").to(device),               output_hidden_states=True).hidden_statesprint("fp16 saglik:", "NaN/Inf VAR -> float32 dene"      if any(torch.isnan(x).any() or torch.isinf(x).any() for x in _h) else "temiz")

## 3. Veri2WikiMultihopQA. **M7:** tarih/sayi varligi iceren sorular elenir — bu adlar gommeuzayinda birbirinden ayirt edilemez ve her iki egriyi de bozar.

In [ ]:
from datasets import load_datasetCANDIDATES = [("xanhho/2WikiMultihopQA","validation"),              ("voidful/2WikiMultihopQA","validation"),              ("hotpotqa/hotpot_qa","validation")]ds = Nonefor cid, sp in CANDIDATES:    try: ds = load_dataset(cid, split=sp); print("YUKLENDI:", cid, "|", len(ds)); break    except Exception as e: print("olmadi:", cid, "->", str(e)[:140])if ds is None: raise RuntimeError("Veri seti yuklenemedi.")print("\n--- alanlar ---")for k, v in ds[0].items(): print(f"{k:20s}: {str(v)[:180]}")

In [ ]:
def to_triples(ex):    ev = ex.get("evidences"); out = []    if ev is None: return out    if isinstance(ev, list):        for e in ev:            if isinstance(e,(list,tuple)) and len(e)==3:                out.append(tuple(str(x).strip() for x in e))            elif isinstance(e, dict):                ks={k.lower():k for k in e}                if {"subject","relation","object"}<=set(ks):                    out.append((str(e[ks["subject"]]).strip(), str(e[ks["relation"]]).strip(),                                str(e[ks["object"]]).strip()))    elif isinstance(ev, dict):        ks={k.lower():k for k in ev}        if {"subject","relation","object"}<=set(ks):            for s,r,o in zip(ev[ks["subject"]],ev[ks["relation"]],ev[ks["object"]]):                out.append((str(s).strip(),str(r).strip(),str(o).strip()))    return outdef find_chain(t):    for i,(s1,r1,o1) in enumerate(t):        for j,(s2,r2,o2) in enumerate(t):            if i!=j and o1 and s2 and o1.lower()==s2.lower() and o2 and o2.lower()!=s1.lower():                return (s1,r1,o1,r2,o2)    return None# M7: tarih / sayi agirlikli adlar gomme uzayinda ayirt edilemezDATEY = re.compile(r"^[\d\s\-/.,]+$|\b(januar|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\w*\b\s*\d",                   re.IGNORECASE)def bad_entity(e):    e = str(e).strip()    if len(e) < 2: return True    digits = sum(c.isdigit() for c in e)    return bool(DATEY.match(e)) or digits > len(e) * 0.4items, drop_date = [], 0for ex in ds:    ty = str(ex.get("type","")).lower()    if ty and ty not in ("compositional","inference","bridge"): continue    ch = find_chain(to_triples(ex))    if not (ch and all(ch)): continue    e1,r1,br,r2,e3 = ch    if bad_entity(br) or bad_entity(e3):        drop_date += 1; continue    items.append(dict(question=str(ex["question"]), e1=e1, r1=r1, bridge=br, r2=r2, e3=e3))    if len(items) >= MAX_ITEMS: breakprint(f"\nzincirli soru: {len(items)}   (tarih/sayi varligi nedeniyle elenen: {drop_date})")if not items: raise RuntimeError("Zincir cikmadi -> to_triples() semayi tutmuyor.")print(json.dumps(items[0], indent=2, ensure_ascii=False))

## 4. Bilgi filtresiModelin **her iki tek-hop olguyu da bildigi** sorular tutulur. Bu adim olmadan olculen seyadresleme degil bilgi olur. Filtre ayni zamanda Asama A icin **%100'luk bir tavan** tanimlar:dogrudan sorulunca model hop-2'yi zaten biliyor.

In [ ]:
from tqdm.auto import tqdmFEW = ("Answer each question with a short factual answer.\n\n"       "Q: What is the director of Inception?\nA: Christopher Nolan\n\n"       "Q: What is the capital of France?\nA: Paris\n\n")def prompt(q): return FEW + "Q: " + q + "\nA:"def norm(s):    s = str(s).lower().strip()    s = re.sub(r"\b(a|an|the)\b"," ",s).translate(str.maketrans("","",string.punctuation))    return " ".join(s.split())def match(pred, gold):    p,g = norm(pred), norm(gold)    if len(g) < 3: return p == g    return p == g or (" "+g+" ") in (" "+p+" ")@torch.no_grad()def gen(prompts, desc, mx=12):    out=[]    for i in tqdm(range(0,len(prompts),GEN_BATCH), desc=desc):        enc = tok(prompts[i:i+GEN_BATCH], return_tensors="pt", padding=True,                  truncation=True, max_length=384).to(device)        g = model.generate(**enc, max_new_tokens=mx, do_sample=False,                           pad_token_id=tok.pad_token_id)        for row in g[:, enc["input_ids"].shape[1]:]:            out.append(tok.decode(row, skip_special_tokens=True).split("\n")[0].strip())    return outif cached("kept.json"):    kept = json.load(open(cpath("kept.json"), encoding="utf-8")); print("onbellek:", len(kept))else:    a1 = gen([prompt(f"What is the {it['r1']} of {it['e1']}?") for it in items], "hop-1")    a2 = gen([prompt(f"What is the {it['r2']} of {it['bridge']}?") for it in items], "hop-2")    kept = [dict(it) for it,x,y in zip(items,a1,a2)            if match(x, it["bridge"]) and match(y, it["e3"])]    json.dump(kept, open(cpath("kept.json"),"w",encoding="utf-8"), ensure_ascii=False, indent=1)print(f"filtre: {len(items)} -> {len(kept)}")if len(kept) == 0: raise RuntimeError("Filtreden hic soru gecmedi.")if len(kept) < MIN_KEPT:    print(f"!! UYARI: {MIN_KEPT} altinda. Daha buyuk model ya da MAX_ITEMS artir.")

## ASAMA A — Oncul testi**Soru:** cok-hop basarisizligi gercekten bir *adresleme* problemi mi?**M1 duzeltmesi.** Kapi artik `dogru ipucu - IPUCUSUZ`. Onceki surumde `dogru - rastgele`kullaniliyordu; rastgele ipucu notr degil **dusmanca** oldugundan (model yanlis varligatutunur) tek basina esigi doldurabiliyordu. Rastgele kol artik **saglamlik kontrolu**:`rastgele <= ipucusuz` olmali, yoksa model ipucunu hic okumuyor demektir.**M6 duzeltmesi.** Iki ipucu formati denenir. Ikisi de tavanin cok altinda kalirsa sonuc"oncul yanlis" degil **"ipucu kullanilamiyor"**dur — tasarimi yanlislikla oldurmemek icin.**K1 duzeltmesi.** Rastgele ipucunun gercek kopruden **string olarak** farkli oldugu garanti.

In [ ]:
def hint_paren(it, b):   return it["question"] + f" (Hint: {b})"def hint_premise(it, b): return f"Given that the {it['r1']} of {it['e1']} is {b}, {it['question']}"if cached("stageA.json"):    A = json.load(open(cpath("stageA.json"))); print("onbellek")else:    rs = random.Random(SEED)    pool = [it["bridge"] for it in kept]    rand_b = []    for it in kept:                                   # K1: string farkliligi garanti        c = pool[rs.randrange(len(pool))]        tries = 0        while norm(c) == norm(it["bridge"]) and tries < 50:            c = pool[rs.randrange(len(pool))]; tries += 1        rand_b.append(c)    runs = {        "none":          [prompt(it["question"])                      for it in kept],        "paren":         [prompt(hint_paren(it, it["bridge"]))        for it in kept],        "premise":       [prompt(hint_premise(it, it["bridge"]))      for it in kept],        "rand_paren":    [prompt(hint_paren(it, b))                   for it, b in zip(kept, rand_b)],        "rand_premise":  [prompt(hint_premise(it, b))                 for it, b in zip(kept, rand_b)],    }    A = {}    for k, ps in runs.items():        o = gen(ps, k)        A[k] = float(np.mean([match(a, it["e3"]) for a, it in zip(o, kept)]))    json.dump(A, open(cpath("stageA.json"),"w"), indent=1)def bprop(p, n, salt=0, nb=N_BOOT):    r = np.random.RandomState((SEED+salt) % (2**31-1)); k = int(round(p*n))    x = np.array([1]*k + [0]*(n-k))    m = x[r.randint(0,n,size=(nb,n))].mean(1)    return float(np.percentile(m,2.5)), float(np.percentile(m,97.5))n = len(kept)# hangi ipucu formati daha iyi calisiyor -> o formatin rastgele kolu ile eslestirfmt = "premise" if A["premise"] >= A["paren"] else "paren"acc_hint, acc_rand, acc_none = A[fmt], A["rand_" + fmt], A["none"]gain  = acc_hint - acc_none          # M1: KAPI BUmis   = acc_rand - acc_none          # rastgele ipucunun etkisi (negatif olmali)print("="*70)print("ASAMA A — her iki tek-hop olgu BILINIYOR kosuluyla, 2-hop dogrulugu")print(f"  (dogrudan sorulunca hop-2 dogrulugu = %100 — filtrenin garantisi, TAVAN)")print("-"*70)for si, k in enumerate(("none","paren","premise","rand_paren","rand_premise")):    lo,hi = bprop(A[k], n, salt=si)   # sabit tohum: hash() surecler arasi degisir    star = "  <- secilen format" if k == fmt else ""    print(f"  {k:14s}: %{100*A[k]:5.1f}   GA [%{100*lo:.1f}, %{100*hi:.1f}]{star}")print("-"*70)print(f"  ADRESLEME KAZANCI (dogru ipucu - ipucusuz) : {100*gain:+.1f} puan   [KAPI: >= {100*GATE_GAIN:+.1f}]")print(f"  rastgele ipucunun etkisi                   : {100*mis:+.1f} puan   [saglamlik: <= 0 olmali]")print(f"  dogru-ipucu tavana yakinlik                : %{100*acc_hint:.1f}   [esik: %{100*HINT_CEILING:.0f}]")print("="*70)HINT_USABLE = acc_hint >= HINT_CEILINGA_PASS      = HINT_USABLE and gain >= GATE_GAINREADS_HINT  = mis <= 0.02if not HINT_USABLE:    STAGE_A = "IPUCU KULLANILAMIYOR"elif A_PASS:    STAGE_A = "GECTI"else:    STAGE_A = "ONCUL YANLIS"print("ASAMA A:", STAGE_A)if STAGE_A == "GECTI":    print("  -> Kopruyu vermek belirgin sekilde yardim ediyor: darbogaz adresleme. Asama B'ye gec.")    if not READS_HINT:        print("  !! NOT: rastgele ipucu zarar vermiyor -> model ipucunu okumuyor olabilir.")        print("     Kazanc baska bir mekanizmadan geliyorsa yorum dikkatli yapilmali.")elif STAGE_A == "IPUCU KULLANILAMIYOR":    print("  -> Model, dogrudan sorulunca hop-2'yi %100 biliyor ama ipucu olarak verilince")    print("     kullanamiyor. Bu ADRESLEME hakkinda bilgi VERMEZ - talimat izleme sorunu.")    print("  EYLEM: once ipucu formatini coz (chat template, few-shot ornegi, decompose).")else:    print("  -> Kopruyu bedava vermek yardim etmiyor: darbogaz adresleme DEGIL.")    print("  EYLEM: DUR. Latent adresli bellek bu problemi cozmez; tasarimi bastan kur.")def require_A():    # M2: kapi artik GERCEKTEN durduruyor    if STAGE_A != "GECTI":        raise RuntimeError(f"ASAMA A gecilemedi ({STAGE_A}). Asama B calistirilmaz — "                           "yukaridaki EYLEM satirina bak.")

## ASAMA B — Cikarilabilirlik ve zamanlama**Neden egitilmis prob:** tasarim ham durumu adres olarak onermiyor, ogrenilmis bir sorgubasligi oneriyor. Ham benzerlikte sinyal gorunmemesi bir sey kanitlamaz. Ridge prob tamdenetimle bile cikaramiyorsa bilgi kullanilabilir bicimde gercekten yoktur —**bu yorumlanabilir bir negatiftir.****M4 duzeltmesi:** iki ayri bolme. Kopru probu **kopru varligina**, cevap probu **cevapvarligina** gore bolunur; her iki egri de kendi hedefinde ezberlemeye karsi korunur.Onceki surumde yalniz kopru korunuyordu ve karar iki korumasiz-olmayan egriyi kiyasliyordu.**M5 duzeltmesi:** varlik gomme katmani artik keyfi degil, **val'da secilen hiperparametre**.

In [ ]:
require_A()@torch.no_grad()def enc_layers(texts, mode, mx, layer_ids):    out = np.zeros((len(texts), len(layer_ids), H), dtype=np.float16)    for i in tqdm(range(0,len(texts),ENC_BATCH), desc=f"kodlama[{mode}]"):        ch = texts[i:i+ENC_BATCH]        e = tok(ch, return_tensors="pt", padding=True, truncation=True, max_length=mx).to(device)        hs = model(**e, output_hidden_states=True).hidden_states        m = e["attention_mask"].unsqueeze(-1).half(); den = m.sum(1).clamp(min=1)        for j,l in enumerate(layer_ids):            v = hs[l][:,-1,:] if mode=="last" else (hs[l]*m).sum(1)/den            out[i:i+len(ch), j, :] = v.float().cpu().numpy().astype(np.float16)    return outent_list = sorted({it["bridge"] for it in kept} | {it["e3"] for it in kept})ent_ix   = {e:i for i,e in enumerate(ent_list)}print("benzersiz varlik:", len(ent_list))if cached("states.npy"):    States = np.load(cpath("states.npy")); EntRaw = np.load(cpath("entemb.npy"))    print("onbellek:", States.shape, EntRaw.shape)else:    States = enc_layers([prompt(it["question"]) for it in kept], "last", 320, LAYERS)    EntRaw = enc_layers(ent_list, "mean", 32, TGT_LAYERS)   # K3: sadece aday katmanlar    np.save(cpath("states.npy"), States); np.save(cpath("entemb.npy"), EntRaw)    print("kaydedildi:", States.shape, EntRaw.shape)# aday hedef uzaylari, normalizeENT = []for ti in range(len(TGT_LAYERS)):    E = EntRaw[:, ti, :].astype(np.float32)    ENT.append(E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-8))y_bridge = np.array([ent_ix[it["bridge"]] for it in kept])y_answer = np.array([ent_ix[it["e3"]]     for it in kept])CAND = np.arange(len(ent_list))

In [ ]:
def make_split(keyfn, salt):    # M4: hedef varliga gore bolme -> test hedefleri egitimde HIC gorulmez    uniq = sorted({keyfn(it) for it in kept})    rs = random.Random(SEED + salt); rs.shuffle(uniq)    n1, n2 = int(.6*len(uniq)), int(.8*len(uniq))    grp = {e:("tr" if i<n1 else "va" if i<n2 else "te") for i,e in enumerate(uniq)}    sp = np.array([grp[keyfn(it)] for it in kept])    TR, VA, TE = sp=="tr", sp=="va", sp=="te"    assert not ({keyfn(kept[i]) for i in np.where(TR)[0]} &                {keyfn(kept[i]) for i in np.where(TE)[0]}), "SIZINTI"    return TR, VA, TE, len(uniq)SPL = {"bridge": make_split(lambda it: it["bridge"], 5),       "answer": make_split(lambda it: it["e3"],     6)}for k,(TR,VA,TE,nu) in SPL.items():    print(f"{k:7s}: benzersiz hedef {nu:4d} -> train {TR.sum():4d} val {VA.sum():4d} test {TE.sum():4d}")    if TR.sum() < MIN_TRAIN:        print(f"  !! UYARI: train {TR.sum()} < {MIN_TRAIN}; PCA_DIM={PCA_DIM} asiri uyum riski.")def pca_basis(Xtr, k):    mu = Xtr.mean(0)    _,_,Vt = np.linalg.svd(Xtr - mu, full_matrices=False)    return mu, Vt[:int(min(k, Vt.shape[0], Xtr.shape[0]-1))].Tdef ridge_solve(G, B, lam):    return np.linalg.solve(G + lam*np.eye(G.shape[0]), B)def nrank(E, P, y_true, cand):    P = P / (np.linalg.norm(P,axis=1,keepdims=True) + 1e-8)    S = P @ E[cand].T    pos = {c:i for i,c in enumerate(cand)}    out = np.empty(len(y_true))    for i, yt in enumerate(y_true):        g = S[i, pos[yt]]        out[i] = ((S[i] > g).sum() + ((S[i] == g).sum()-1)/2) / max(1, len(cand)-1)    return outdef bci(x, salt=0, nb=N_BOOT):    x = np.asarray(x,float); x = x[np.isfinite(x)]    if len(x) < 5: return (np.nan, np.nan)    r = np.random.RandomState((SEED+salt) % (2**31-1))    m = np.median(x[r.randint(0,len(x),size=(nb,len(x)))],axis=1)    return float(np.percentile(m,2.5)), float(np.percentile(m,97.5))

In [ ]:
require_A()res, raw, hyper = {}, {}, {}for tgt, y in (("bridge", y_bridge), ("answer", y_answer)):    TR, VA, TE, _ = SPL[tgt]    res[tgt], raw[tgt] = {}, {}    for j, L in enumerate(tqdm(LAYERS, desc=f"katman[{tgt}]")):        X = States[:, j, :].astype(np.float32)        mu, V = pca_basis(X[TR], PCA_DIM)              # katman basina TEK SVD        Ztr, Zva, Zte = (X[TR]-mu)@V, (X[VA]-mu)@V, (X[TE]-mu)@V        G = Ztr.T @ Ztr        best = None        for ti, E in enumerate(ENT):                   # M5: hedef katman = hiperparametre            B = Ztr.T @ E[y[TR]]            for lam in LAMBDAS:                A_ = ridge_solve(G, B, lam)                mv = float(np.median(nrank(E, Zva @ A_, y[VA], CAND)))   # VAL'da secim                if best is None or mv < best[0]: best = (mv, A_, ti, lam)        _, A_, ti, lam = best        hyper[(tgt, L)] = (TGT_LAYERS[ti], lam)        res[tgt][L] = nrank(ENT[ti], Zte @ A_, y[TE], CAND)              # TEST'te rapor        # ham kontrol: projeksiyon YOK, uzaylar hizasiz. Kasten zayif — eski tasarimin olctugu sey.        raw[tgt][L] = nrank(ENT[ti], X[TE], y[TE], CAND)np.savez_compressed(cpath("probe.npz"), layers=np.array(LAYERS),                    **{f"{t}_{k}": np.array([d[t][L] for L in LAYERS])                       for t in ("bridge","answer") for k,d in (("probe",res),("raw",raw))})print("kaydedildi ->", cpath("probe.npz"))# K2: secilen hiperparametreler artik raporlaniyorfor tgt in ("bridge","answer"):    tl = [hyper[(tgt,L)][0] for L in LAYERS]; lm = [hyper[(tgt,L)][1] for L in LAYERS]    print(f"{tgt:7s}: hedef katman dagilimi {dict((x,tl.count(x)) for x in sorted(set(tl)))}"          f" | lambda {dict((x,lm.count(x)) for x in sorted(set(lm)))}")if all(hyper[(t,L)][1] == max(LAMBDAS) for t in ("bridge","answer") for L in LAYERS):    print("!! Lambda hep en buyuk -> prob asiri uyum sinirinda; n_train kucuk olabilir.")

## KararKatman, hedef uzayi ve lambda **val**'da secildi; asagidaki tum sayilar **test** setinde.Her iki hedefte de test varliklari egitimde hic gorulmedi.**M3 duzeltmesi:** pencere genisligi karara girdi. `L_kopru < L_cevap` tek basina yetmez —arada is yapilabilecek kadar katman olmali.

In [ ]:
require_A()mb = np.array([np.median(res["bridge"][L]) for L in LAYERS])ma = np.array([np.median(res["answer"][L]) for L in LAYERS])Lb, La = LAYERS[int(np.argmin(mb))], LAYERS[int(np.argmin(ma))]cb, ca = bci(res["bridge"][Lb], 11), bci(res["answer"][La], 12)rb = float(np.median(raw["bridge"][Lb]))win = (La - Lb) / max(1, N_LAYERS)print("="*70)print(f"model {MODEL_ID} | aday varlik {len(ent_list)} | sans 0.500 (asagi = iyi)")print(f"test sorusu: kopru {SPL['bridge'][2].sum()}  cevap {SPL['answer'][2].sum()}")print("-"*70)print(f"  KOPRU (e2)  en iyi katman {Lb:3d}  medyan {mb.min():.3f}  GA [{cb[0]:.3f},{cb[1]:.3f}]")print(f"  CEVAP (e3)  en iyi katman {La:3d}  medyan {ma.min():.3f}  GA [{ca[0]:.3f},{ca[1]:.3f}]")print(f"  ham kontrol (projeksiyonsuz, katman {Lb}): {rb:.3f}")print(f"  PENCERE: L{Lb} -> L{La} = {La-Lb} katman  (%{100*win:.1f} derinlik)"      f"   [esik: %{100*MIN_WINDOW_FRAC:.0f}]")print("="*70)bridge_found = np.isfinite(cb[1]) and cb[1] < BRIDGE_CI_MAXearlier      = Lb < Lawide_enough  = win >= MIN_WINDOW_FRACif   not bridge_found: R = "ADRES YOK"elif not earlier:      R = "KISAYOL"elif not wide_enough:  R = "PENCERE DAR"else:                  R = "GEC"print("SONUC:", R, "\n")if R == "ADRES YOK":    print("Ezberleyemeyen, tam denetimli bir prob bile kopruyu cikaramiyor.")    print("Kopru kimligi latent durumda DOGRUSAL olarak kullanilabilir bicimde yok.")    print("EYLEM: kapsam degisir - bu bir lookup-tablosu projesi degil. Kopruyu temsile")    print("       sokmak egitim/mimari degisikligi gerektirir. (Dogrusal olmayan bir")    print("       baslik bulabilir; bu yuzden iptal degil, kapsam degisikligi.)")elif R == "KISAYOL":    print(f"Kopru L{Lb}'de, cevap L{La}'de - kopru cevaptan ONCE cozulmuyor.")    print("Model muhtemelen e1->e3 kisayolunu kullaniyor; ayrik bir ara adim yok.")    print("EYLEM: adresleyecek bir ara adim mevcut degil. Once kisayol kullanmayan")    print("       ornekler (nadir cevap / sik kopru) ile tekrarla.")elif R == "PENCERE DAR":    print(f"Kopru cevaptan once cozuluyor ama arada yalnizca {La-Lb} katman var (%{100*win:.1f}).")    print("Adres mevcut, ama onu kullanip bellekten bir sey getirecek hesap zamani yok.")    print("EYLEM: donguye baglamanin faydasi supheli. Once daha derin / looped bir")    print("       backbone'da tekrarla (pencere derinlikle acilabilir).")else:    print(f"Kopru L{Lb}'de cozuluyor, cevap L{La}'de - arada {La-Lb} katman (%{100*win:.1f}).")    print("Hem adres var hem de kullanilabilecek genislikte bir pencere var.")    print("EYLEM: Deney 2 - projeksiyonu donguye bagla, merdiven testi")    print("       (R_max=1..4 sinirliyken k-hop dogrulugu).")if rb > 0.45 and bridge_found:    print(f"\nNOT: ham kontrol {rb:.3f} (sansta) iken egitilmis prob {mb.min():.3f}.")    print("     Projeksiyonsuz bir tasarim burada YANLIS NEGATIF verirdi.")json.dump(dict(model=MODEL_ID, stage_a=STAGE_A, result=R, A=A, hint_fmt=fmt,               gain=float(gain), mis=float(mis), reads_hint=bool(READS_HINT),               n_kept=len(kept), n_ent=len(ent_list),               L_bridge=int(Lb), L_answer=int(La), window_frac=float(win),               med_bridge=float(mb.min()), ci_bridge=list(cb),               med_answer=float(ma.min()), ci_answer=list(ca), raw_bridge=rb),          open(cpath("verdict.json"),"w"), indent=1)print("\nkaydedildi ->", cpath("verdict.json"))

## Grafik

In [ ]:
require_A()import matplotlib.pyplot as pltfig, ax = plt.subplots(figsize=(11,5.5))ax.plot(LAYERS, mb, "-o", color="tab:red",  lw=2, ms=4, label="KOPRU (e2) — egitilmis prob")ax.plot(LAYERS, ma, "-o", color="tab:blue", lw=2, ms=4, label="CEVAP (e3) — egitilmis prob")ax.plot(LAYERS, [np.median(raw["bridge"][L]) for L in LAYERS], "--",        color="tab:gray", lw=1.5, label="kopru, HAM (projeksiyonsuz kontrol)")ax.axhline(0.5, ls="--", color="k", lw=1, label="sans")ax.axvline(Lb, color="tab:red", ls=":", lw=1.5); ax.axvline(La, color="tab:blue", ls=":", lw=1.5)if La > Lb:    ax.axvspan(Lb, La, color="tab:green", alpha=0.10)    ax.text((Lb+La)/2, 0.06, f"pencere %{100*win:.0f}", ha="center", fontsize=9, color="tab:green")ax.set_ylim(0,1); ax.invert_yaxis(); ax.grid(alpha=0.3)ax.set_xlabel("katman"); ax.set_ylabel("medyan normalize sira (asagi = iyi)")ax.set_title(f"{MODEL_ID} — test seti; her iki hedefte de test varliklari egitimde GORULMEDI")ax.legend(loc="lower left", fontsize=9)plt.tight_layout(); plt.savefig(cpath("kopru.png"), dpi=150); plt.show()

## Sinirlar — bilerek yapilmayanlar- **Derinlik, dongu adiminin tam vekili degildir.** Jacobian Lens (arXiv 2609.01924) looped  modellerde workspace'in her donguda yeniden kuruldugunu (Ouro) ya da dar bir pencerede  tasindigini (Huginn) buluyor. Buradaki `GEC` sonucu Deney 2'yi **gerekceler**, garanti etmez.- **Dogrusal prob.** `ADRES YOK`, "dogrusal olarak cikarilamiyor" demektir; dogrusal olmayan  bir baslik bulabilir. Bu yuzden o sonuc **kapsam degisikligi** onerir, iptal degil.- **Iki egri iki ayri bolmede olculur** (M4). Her biri kendi hedefinde durusttur; ama ayni  soru altkumesinde degildirler. Argmin karsilastirmasi bu nedenle katman-duzeyinde  gecerlidir, soru-duzeyinde eslesmis degildir.- **Esikler secildi:** `GATE_GAIN=0.15`, `HINT_CEILING=0.60`, `BRIDGE_CI_MAX=0.45`,  `MIN_WINDOW_FRAC=0.15`. Sonuc sinirda kalirsa tartisilacak sey esiktir, tasarim degil.- Sonuc **bu model** icindir. Ikinci bir `MODEL_ID` ile tekrarla; onbellek damgalidir.### Kaydedilenler`kept.json` · `stageA.json` · `states.npy` · `entemb.npy` · `probe.npz` ·`verdict.json` · `kopru.png`### Sorun giderme- **Asama A "IPUCU KULLANILAMIYOR" dedi:** model talimat izleyemiyor. Daha buyuk model dene  ya da chat template + few-shot ornegi ekle. Bu bir tasarim sonucu DEGILDIR.- **Filtreden az soru geciyor:** daha buyuk model ya da `MAX_ITEMS` artir.- **train < MIN_TRAIN uyarisi:** `PCA_DIM` dusur (or. 128).- **Veri seti:** `CANDIDATES` + `to_triples()`.